# 🔬 llama.cpp 源码级深度剖析

> **核心命题**：如何用纯 C/C++ 在消费级硬件上跑 Llama、Mistral、DeepSeek 等模型？

llama.cpp 是整个 LLM 推理生态中最特殊的项目——没有之一。它不依赖 PyTorch，不依赖 CUDA Toolkit（可选），
核心推理引擎 `ggml` 只有 ~70K 行 C 代码。理解它就理解了推理的最小闭环。

## 📐 架构全景

```
┌──────────────────────────────────────────────────────────────┐
│                       llama.cpp 架构                          │
├──────────────────────────────────────────────────────────────┤
│                                                               │
│  ┌─────────────┐    ┌─────────────┐    ┌─────────────────┐  │
│  │  llama.cpp   │    │  server/    │    │  examples/      │  │
│  │  (高层 API)  │    │  (HTTP API) │    │  (应用层)       │  │
│  └──────┬───────┘    └──────┬──────┘    └────────┬────────┘  │
│         │                  │                     │           │
│         └──────────────────┼─────────────────────┘           │
│                            │                                 │
│  ┌─────────────────────────▼─────────────────────────────┐  │
│  │              llama.h / llama.cpp                       │  │
│  │  • llama_model / llama_context 结构体                  │  │
│  │  • 模型加载（GGUF → 内存）                             │  │
│  │  • 推理循环（prefill → decode → sample）               │  │
│  │  • KV Cache 管理                                       │  │
│  └─────────────────────────┬─────────────────────────────┘  │
│                            │                                 │
│  ┌─────────────────────────▼─────────────────────────────┐  │
│  │                   ggml (核心计算库)                     │  │
│  │  • 计算图（compute graph）构建与执行                    │  │
│  │  • 张量操作（矩阵乘法、注意力、归一化...）              │  │
│  │  • 后端抽象层：CPU / CUDA / Metal / Vulkan / SYCL     │  │
│  │  • 量化算子（dequantize → compute → quantize）         │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                               │
└──────────────────────────────────────────────────────────────┘
```

### 关键源文件映射

| 文件 | 职责 | 行数(约) |
|------|------|----------|
| `ggml/include/ggml.h` | ggml 核心 API 与张量定义 | 7000+ |
| `ggml/src/ggml.c` | 计算图构建 + CPU 后端计算内核 | 18000+ |
| `ggml/src/ggml-cuda/` | CUDA 后端（含 cuBLAS, FlashAttention） | 多文件 |
| `ggml/src/ggml-metal/` | Apple Metal 后端 | 3000+ |
| `gguf/gguf.py` | Python 端 GGUF 读写（参考实现） | 500+ |
| `llama.cpp` | 模型加载 + 推理循环主逻辑 | 18000+ |
| `llama.h` | 公开 C API | 800+ |

本文按照 **GGUF 格式 → ggml 计算图 → 推理循环 → 量化细节** 的顺序，
从数据流视角追踪一个 token 的完整生命周期。

## 1. GGUF 格式：模型文件的基因编码

GGUF (GGML Universal Format) 是 llama.cpp 的模型文件格式，取代了早期的 GGML 格式。
它的核心设计目标：**一次写入，任意平台读取，无需 pickle**。

### 1.1 为什么不用 safetensors？

| | GGUF | safetensors |
|------|------|-------------|
| 量化支持 | **原生内置**（Q4_0, Q4_K_M, ...） | 需要外部量化方案 |
| 元数据 | 内嵌在文件头（Tokenizer, chat template, LORA config） | 分离的 `config.json` |
| 跨平台 | 单一文件，extensible key-value | 需要 HuggingFace hub 配套 |
| mmap 加载 | **设计目标** — 零拷贝直接映射 | 需要先解析再加载 |

### 1.2 文件布局（源码 gguf/gguf.py）

```
┌──────────────────────────────────────┐
│  GGUF Magic: "GGUF" (4 bytes)       │ ← 0x00
├──────────────────────────────────────┤
│  Version: uint32 (当前 v3)            │ ← 0x04
├──────────────────────────────────────┤
│  Tensor count: uint64                │ ← 0x08
├──────────────────────────────────────┤
│  Metadata KV count: uint64           │ ← 0x10
├──────────────────────────────────────┤
│  Metadata K-V pairs:                 │
│    key: string                       │
│    value_type: gguf_type (uint32)    │
│    value: ...                        │
│  ... (循环 NKV 次)                    │
│  关键 metadata:                      │
│    "general.architecture": "llama"   │
│    "llama.context_length": 8192      │
│    "llama.embedding_length": 4096    │
│    "llama.block_count": 32           │
│    "llama.feed_forward_length": 11008│
│    "llama.attention.head_count": 32  │
│    "tokenizer.ggml.tokens": [...]    │
├──────────────────────────────────────┤
│  Tensor info (× tensor_count):       │
│    name: string                      │
│    n_dims: uint32                    │
│    shape[n_dims]: uint64[]           │
│    type: ggml_type (uint32)          │ ← Q4_K_M / F16 / ...
│    offset: uint64                    │ ← 在文件中的偏移
├──────────────────────────────────────┤
│  Padding (对齐到 alignment)           │
├──────────────────────────────────────┤
│  Tensor data (× tensor_count):       │
│    block 0: [weight data in type]    │
│    block 1: [weight data in type]    │
│    ...                               │
└──────────────────────────────────────┘
```

**关键洞察**：metadata 在 tensor data 之前 — 这意味着你不需要读取整个文件就能知道模型结构。
加载时用 `mmap` 映射 tensor data 区域，OS 按需分页加载，实现「零拷贝」。

In [ ]:
# 实验：解析 GGUF 文件头
# 下载一个小的 GGUF 模型（或用 ollama 已有的）来实验

import struct, os, sys
from pathlib import Path

def parse_gguf_header(filepath: str):
    """解析 GGUF 文件头，对应 gguf/gguf.py 的 GGUFReader"""
    with open(filepath, 'rb') as f:
        # Magic
        magic = f.read(4)
        if magic != b'GGUF':
            raise ValueError(f"Not a GGUF file: magic={magic}")
        
        # Version
        version = struct.unpack('<I', f.read(4))[0]
        
        # Counts
        tensor_count = struct.unpack('<Q', f.read(8))[0]
        kv_count = struct.unpack('<Q', f.read(8))[0]
        
        print(f"GGUF v{version}")
        print(f"Tensors: {tensor_count}")
        print(f"Metadata keys: {kv_count}")
        print()
        
        # Parse metadata k-v pairs
        GGUF_TYPES = {
            0: 'u8', 1: 'i8', 2: 'u16', 3: 'i16', 4: 'u32', 5: 'i32',
            6: 'f32', 7: 'bool', 8: 'string', 9: 'array', 10: 'u64', 11: 'i64',
            12: 'f64'
        }
        
        for i in range(kv_count):
            # Read string key
            key_len = struct.unpack('<Q', f.read(8))[0]
            key = f.read(key_len).decode('utf-8')
            
            # Read value type
            val_type = struct.unpack('<I', f.read(4))[0]
            type_name = GGUF_TYPES.get(val_type, f'unknown({val_type})')
            
            # Read value based on type (简化处理，只显示摘要)
            if val_type == 8:  # string
                vlen = struct.unpack('<Q', f.read(8))[0]
                val = f.read(vlen).decode('utf-8', errors='replace')
                if len(val) > 60:
                    val = val[:60] + '...'
            elif val_type in (0, 1, 2, 3, 4, 5, 10, 11):  # integers
                size_map = {0:1, 1:1, 2:2, 3:2, 4:4, 5:4, 10:8, 11:8}
                val = struct.unpack({1:'b',2:'h',4:'i',8:'q'}[size_map[val_type]],
                                   f.read(size_map[val_type]))[0]
            elif val_type == 6:  # f32
                val = struct.unpack('<f', f.read(4))[0]
            elif val_type == 7:  # bool
                val = bool(f.read(1)[0])
            elif val_type == 9:  # array
                arr_type = struct.unpack('<I', f.read(4))[0]
                arr_len = struct.unpack('<I', f.read(4))[0]
                val = f"[{GGUF_TYPES.get(arr_type, '?')}] x {arr_len}"
                # skip the actual array data
                elem_size = {0:1,1:1,2:2,3:2,4:4,5:4,6:4,7:1,8:1,10:8,11:8,12:8}
                f.seek(arr_len * elem_size.get(arr_type, 1), 1)
            else:
                val = f"<type {val_type}>"
            
            if 'token' not in key.lower() or 'architecture' in key.lower():
                print(f"  [{type_name}] {key}: {val}")
        
        print(f"\n... (剩余 {kv_count - i - 1} 个 metadata keys)")
        
        # Parse tensor infos (scout first 5)
        GGML_TYPES = {
            0: 'F32', 1: 'F16', 2: 'Q4_0', 3: 'Q4_1',
            6: 'Q5_0', 7: 'Q5_1', 8: 'Q8_0', 9: 'Q8_1',
            10: 'Q2_K', 11: 'Q3_K', 12: 'Q4_K', 13: 'Q5_K',
            14: 'Q6_K', 15: 'Q8_K',
            16: 'IQ2_XXS', 17: 'IQ2_XS', 18: 'IQ3_XXS',
            19: 'IQ1_S', 20: 'IQ4_NL', 21: 'IQ3_S', 22: 'IQ2_S', 23: 'IQ4_XS',
            24: 'I8', 25: 'I16', 26: 'I32', 27: 'I64',
            28: 'F64', 29: 'IQ1_M',
            30: 'BF16', 31: 'Q4_0_4_4', 32: 'Q4_0_4_8', 33: 'Q4_0_8_8',
        }
        
        print(f"\n--- Tensor Info (first 8) ---")
        for i in range(min(tensor_count, 8)):
            name_len = struct.unpack('<Q', f.read(8))[0]
            name = f.read(name_len).decode('utf-8')
            n_dims = struct.unpack('<I', f.read(4))[0]
            shape = [struct.unpack('<Q', f.read(8))[0] for _ in range(n_dims)]
            ggml_type = struct.unpack('<I', f.read(4))[0]
            offset = struct.unpack('<Q', f.read(8))[0]
            type_name = GGML_TYPES.get(ggml_type, f'type_{ggml_type}')
            print(f"  {name}: shape={shape}, type={type_name}, offset={offset}")

print("将上面的 parse_gguf_header() 指向你的 .gguf 文件即可运行")
print("Tip: ollama 下载的模型通常在 ~/.ollama/models/blobs/ 下")


## 2. ggml：自研张量计算引擎

ggml 是整个 llama.cpp 的地基。它的核心抽象非常简洁：

### 2.1 核心数据结构

```c
// ggml/include/ggml.h (简化)

struct ggml_tensor {
    enum ggml_type    type;      // F32, F16, Q4_K_M, ...
    enum ggml_backend  backend;  // CPU, GPU(CUDA), GPU(Metal)
    struct ggml_backend_buffer *buffer;  // 实际存储
    
    int     n_dims;              // 维度数
    int64_t ne[GGML_MAX_DIMS];   // 各维度大小 (GGML_MAX_DIMS=4)
    size_t  nb[GGML_MAX_DIMS];   // 各维度的 stride (字节)
    
    // 计算图连接
    enum ggml_op op;             // 操作类型: GGML_OP_MUL_MAT, GGML_OP_ADD, ...
    struct ggml_tensor *src[GGML_MAX_SRC];  // 输入张量 (最多 3 个)
    struct ggml_tensor *view_src;           // view 的源张量
    
    void *data;                  // 实际数据指针
    char  padding[48];          // 对齐到 cache line
};
```

**设计哲学**：
- 计算图是**惰性构建**的：调用 `ggml_mul_mat(ctx, A, B)` 只是创建一个 ggml_tensor 节点并连接 src
- 实际计算在 `ggml_graph_compute()` 中执行，此时遍历图并调度后端
- `nb[]` stride 机制让 reshape/view/permute 可以不拷贝数据实现

### 2.2 计算图构建 → 执行的全流程

```c
// 简化版本的 llama.cpp 推理中 ggml 的使用流程

// 1. 初始化上下文
struct ggml_init_params params = {
    .mem_size   = 16*1024*1024,  // 16MB scratch buffer
    .mem_buffer = NULL,          // 让 ggml 自己分配
    .no_alloc   = false,
};
struct ggml_context *ctx = ggml_init(params);

// 2. 构建计算图 (惰性，不计算)
struct ggml_tensor *input  = ggml_new_tensor_1d(ctx, GGML_TYPE_F32, n_embd);
struct ggml_tensor *weight = model->layers[0].wq;  // 从模型文件加载的张量
struct ggml_tensor *result = ggml_mul_mat(ctx, weight, input);
// result->op = GGML_OP_MUL_MAT
// result->src[0] = weight, result->src[1] = input

// 3. 构建 cgraph (收集所有需要计算的节点)
struct ggml_cgraph *gf = ggml_new_graph(ctx);
ggml_build_forward_expand(gf, result);
// 这会递归地将 result 及其所有依赖加入 gf

// 4. 执行
ggml_graph_compute_with_ctx(ctx, gf, n_threads);
// 此时才真正触发计算
```

In [ ]:
# 模拟 ggml 的计算图构建过程
# 理解「惰性计算图」vs「eager execution」

from dataclasses import dataclass, field
from typing import List, Optional, Dict
from enum import Enum
import numpy as np

class Op(Enum):
    MUL_MAT = "mul_mat"
    ADD = "add"
    MUL = "mul"
    ROPE = "rope"
    SOFTMAX = "softmax"

@dataclass
class Tensor:
    """对应 ggml_tensor 的简化 Python 模型"""
    name: str
    shape: tuple
    op: Optional[Op] = None      # None = 输入/参数张量
    src: List['Tensor'] = field(default_factory=list)
    data: Optional[any] = None    # 实际数据（eager mode 用）
    
    def compute(self):
        """递归求值（模拟 ggml_graph_compute）"""
        if self.op is None:
            return self.data  # 参数张量
        
        # 先递归计算所有依赖
        src_data = [s.compute() for s in self.src]
        
        # 模拟计算
        print(f"  compute: {self.name} = {self.op.value}({', '.join(s.name for s in self.src)})")
        
        if self.op == Op.MUL_MAT:
            return np.dot(src_data[0], src_data[1])
        elif self.op == Op.ADD:
            return src_data[0] + src_data[1]

# 构建一个简化版的 Attention 计算图
# Q = input @ Wq, K = input @ Wk, V = input @ Wv
input_tok = Tensor("input", (1, 64), data=np.random.randn(1, 64).astype(np.float32))
Wq = Tensor("Wq", (64, 64), data=np.random.randn(64, 64).astype(np.float32))
Wk = Tensor("Wk", (64, 64), data=np.random.randn(64, 64).astype(np.float32))
Wv = Tensor("Wv", (64, 64), data=np.random.randn(64, 64).astype(np.float32))

# 惰性构建计算图
Q = Tensor("Q", (1, 64), op=Op.MUL_MAT, src=[input_tok, Wq])
K = Tensor("K", (1, 64), op=Op.MUL_MAT, src=[input_tok, Wk])
V = Tensor("V", (1, 64), op=Op.MUL_MAT, src=[input_tok, Wv])

print("计算图已构建（尚未执行任何计算）")
print(f"Q 节点: {Q.op} ← {[s.name for s in Q.src]}")
print()

# 执行
print("开始执行计算图:")
q_val = Q.compute()
print(f"Q 形状: {q_val.shape}")

print("\n这就是 ggml_graph_compute() 做的事——")
print("遍历计算图，按拓扑序调用后端算子。")


## 3. 量化：如何在 4-bit 精度下保持可用？

llama.cpp 的量化不是简单的 round-to-nearest，它有自己的一套「K-quant」体系。

### 3.1 为什么简单量化会失败

LLM 权重分布有一个关键特点：**存在 outlier channel**。某些 channel 的权重值域远大于其他。
如果用全局 scale 做 4-bit 量化，outlier channel 会支配 scale，导致正常 channel 的信息被淹没。

```
Naive 4-bit (全局 scale):
  channel 0: [-0.5, 0.3, -0.2, 0.4]  → scale=0.5  → [4, 10, 7, 12]  ← 正常
  channel 99: [-8.2, 5.1, -6.3, 7.8] → scale=8.3  → [0, 9, 2, 10]   ← 还行
  全局 scale = max(0.5, 8.3) = 8.3
  channel 0 在全局 scale 下: [-0.5/8.3*7, ...] → [0, 0, 0, 0]  ← 全变成 0！信息丢失！

K-quant 方案（per-block 量化）:
  每个 super-block (256个权重) 有自己的 scale
  scale 本身也量化（用 FP16 或更小的 6-bit）
  → 解决了 outlier channel 问题
```

### 3.2 K-quant 的 block 结构（以 Q4_K_M 为例）

这是目前最常用的量化格式，在速度和质量间取得了好的平衡。

```
Q4_K_M 文件格式布局 (每个 super-block = 256 个权重):

┌──────────────────────────────────────────────┐
│  Super-block #0 (256 weights → 256 × 4bit)  │
├──────────────────────────────────────────────┤
│  d (FP16, 2 bytes):   大 scale              │  ← 浮点 scale
│  dmin (FP16, 2 bytes): 小 scale              │  ← 用于不对称量化
├──────────────────────────────────────────────┤
│  Sub-block #0 (16 weights × 4bit = 8 bytes) │
│    scales[0] (6-bit)                         │  ← 子块的小 scale
│  Sub-block #1 (16 weights × 4bit = 8 bytes) │
│  ...                                         │
│  Sub-block #15 (16 weights × 4bit = 8 bytes)│
│  → 16 × 16 = 256 weights = 128 bytes        │
│  + scales: 16 × 6-bit = 12 bytes            │
├──────────────────────────────────────────────┤
│  总计: 2 + 2 + 128 + 12 = 144 bytes         │
│  原始 FP16: 256 × 2 = 512 bytes             │
│  压缩比: 512/144 = 3.56×                     │
│  等效 bits/weight: (144 × 8) / 256 = 4.5 bit│
└──────────────────────────────────────────────┘

其他常见格式对比：
  Q4_0:   对称量化，per-block scale=FP16，   4.50 bpw，速度最快
  Q4_K_M: K-quant，6-bit scales，           4.50 bpw，质量好  ← 最推荐
  Q4_K_S: K-quant small，更少子块，          4.25 bpw，体积更小
  Q6_K:   6-bit K-quant，                   6.56 bpw，近乎无损
  Q8_0:   8-bit 对称量化，                   8.50 bpw，接近 FP16 质量
  IQ4_NL: 重要性感知量化（importance-aware），4.25 bpw，新 SOTA
```

In [ ]:
# 实验：模拟 K-quant vs 全局量化的精度差异
import numpy as np

def demo_quantization_quality():
    """演示量化对推理精度的影响"""
    # 模拟一个带有 outlier 的权重向量 (类似 LLM 的真实分布)
    np.random.seed(42)
    n = 256
    # 90% 的 channel 值域正常
    weights_normal = np.random.randn(230).astype(np.float32) * 0.3
    # 10% 的 channel 是 outlier
    weights_outlier = np.random.randn(26).astype(np.float32) * 5.0
    weights = np.concatenate([weights_normal, weights_outlier])
    np.random.shuffle(weights)
    
    print(f"原始权重分布: min={weights.min():.3f}, max={weights.max():.3f}")
    print(f"std={weights.std():.3f}, 99%分位={np.percentile(np.abs(weights), 99):.3f}")
    
    # 方案 A: 全局 4-bit 量化 (naive)
    max_abs = np.abs(weights).max()
    scale_global = max_abs / 7.0  # 4-bit 范围: -8 ~ +7
    q_global = np.clip(np.round(weights / scale_global), -8, 7).astype(np.int8)
    dq_global = q_global.astype(np.float32) * scale_global
    mse_global = np.mean((weights - dq_global) ** 2)
    
    # 方案 B: per-block 4-bit (K-quant 的方式)
    block_size = 32
    dq_block = np.zeros_like(weights)
    for i in range(0, n, block_size):
        block = weights[i:i+block_size]
        max_abs_b = np.abs(block).max()
        scale_b = max_abs_b / 7.0 if max_abs_b > 0 else 1.0
        q_b = np.clip(np.round(block / scale_b), -8, 7).astype(np.int8)
        dq_block[i:i+block_size] = q_b.astype(np.float32) * scale_b
    mse_block = np.mean((weights - dq_block) ** 2)
    
    print(f"\n全局 4-bit 量化 MSE: {mse_global:.6f}")
    print(f"Per-block 4-bit 量化 MSE:  {mse_block:.6f}")
    print(f"提升: {mse_global/mse_block:.1f}x")
    
    # 对比 outlier channel 的量化效果
    outlier_idx = np.argmax(np.abs(weights))
    print(f"\nOutlier channel (idx={outlier_idx}, val={weights[outlier_idx]:.3f}):")
    print(f"  全局: dequant = {dq_global[outlier_idx]:.3f}")
    print(f"  分块: dequant = {dq_block[outlier_idx]:.3f}")
    
    # 看一个正常 channel
    normal_idx = np.argmin(np.abs(weights - 0.1))
    print(f"\nNormal channel (idx={normal_idx}, val={weights[normal_idx]:.3f}):")
    print(f"  全局: dequant = {dq_global[normal_idx]:.3f} (误差 = {abs(weights[normal_idx]-dq_global[normal_idx]):.4f})")
    print(f"  分块: dequant = {dq_block[normal_idx]:.3f} (误差 = {abs(weights[normal_idx]-dq_block[normal_idx]):.4f})")

demo_quantization_quality()


## 4. 推理循环：一个 Token 的完整旅程

这是 llama.cpp 最核心的流程。我们从 `llama_decode` 入口函数追踪一个 token 从输入到输出的全过程。

### 4.1 顶层调用路径

```c
// 用户调用的 API 链 (main 函数或 server 中)
llama_decode(ctx, batch)
  └─> llama_decode_internal(lctx, batch)
        ├─> 阶段 1: Prefill (如果是新序列)
        │    └─> 处理整个 prompt，填充 KV Cache
        │
        └─> 阶段 2: Decode (自回归生成)
             ├─> 构建 ggml 计算图 (build_llama_graph)
             ├─> 分割计算图以适应后端 (ggml_backend_sched)
             ├─> 执行计算 (ggml_backend_graph_compute)
             └─> 采样下一个 token (llama_sampler_sample)
```

### 4.2 build_llama_graph：构建 Transformer 计算图

这是 llama.cpp 中最重要的函数（~1200 行），它在 ggml 的惰性计算图上构建完整的 Transformer 前向传播。

```c
// 简化版 build_llama_graph（对应 llama.cpp 源码）
static struct ggml_cgraph * build_llama_graph(
    llama_context & lctx,
    const llama_batch & batch
) {
    // ===== 1. Embedding =====
    cur = llm_build_inp_embd(ctx0, lctx, hparams, batch, model);
    
    // ===== 2. Transformer Layers (循环) =====
    for (int il = 0; il < n_layer; il++) {
        struct ggml_tensor * inpL = cur;
        
        // 2a. RMS Norm (pre-norm)
        cur = ggml_rms_norm(ctx0, inpL, norm_eps);
        cur = ggml_mul(ctx0, cur, model->layers[il].attn_norm);
        
        // 2b. Self-Attention
        struct ggml_tensor * Q = ggml_mul_mat(ctx0, wq, cur);
        struct ggml_tensor * K = ggml_mul_mat(ctx0, wk, cur);
        struct ggml_tensor * V = ggml_mul_mat(ctx0, wv, cur);
        
        // RoPE (旋转位置编码)
        Q = ggml_rope(ctx0, Q, n_past, n_rot, 0);
        K = ggml_rope(ctx0, K, n_past, n_rot, 0);
        
        // KV Cache 存储 (关键！)
        ggml_build_forward_expand(gf, 
            ggml_cpy(ctx0, K, kv_cache.k_l[il]));  // 存储 K
        ggml_build_forward_expand(gf,
            ggml_cpy(ctx0, V, kv_cache.v_l[il]));  // 存储 V
        
        // 读取完整 KV Cache (当前 token + 历史)
        struct ggml_tensor * Kq = ggml_view_2d(ctx0, kv_cache.k_l[il], 
                                              n_embd_head, n_tokens);
        struct ggml_tensor * Vq = ggml_view_2d(ctx0, kv_cache.v_l[il],
                                              n_tokens, n_embd_head);
        
        // Attention: softmax(Q @ K^T / sqrt(d)) @ V
        cur = ggml_mul_mat(ctx0, Vq, 
              ggml_soft_max(ctx0,
                ggml_mul_mat(ctx0, Kq, Q)));
        
        // 2c. Output projection + residual
        cur = ggml_mul_mat(ctx0, wo, cur);
        cur = ggml_add(ctx0, cur, inpL);
        
        // 2d. Feed-Forward Network (SwiGLU)
        struct ggml_tensor * ffn_inp = cur;
        cur = ggml_rms_norm(ctx0, cur, norm_eps);
        // gate_proj → silu → * up_proj → down_proj
        struct ggml_tensor * gate = ggml_silu(ctx0, 
            ggml_mul_mat(ctx0, w3, cur));
        struct ggml_tensor * up = ggml_mul_mat(ctx0, w1, cur);
        cur = ggml_mul(ctx0, gate, up);
        cur = ggml_mul_mat(ctx0, w2, cur);
        cur = ggml_add(ctx0, cur, ffn_inp);
    }
    
    // ===== 3. Final RMS Norm + LM Head =====
    cur = ggml_rms_norm(ctx0, cur, norm_eps);
    cur = ggml_mul_mat(ctx0, model->output, cur);  // logits
    
    return gf;
}
```

**关键观察**：
1. 整个 Transformer 只是 ggml 操作的一张**惰性计算图**——构建时零计算
2. KV Cache 的存储/读取是通过**同一个 ggml_tensor 的不同 view** 实现的，无需拷贝
3. 计算图构建完毕后，`ggml_backend_sched` 会按后端能力**分割计算图**——MatMul → GPU，Norm → CPU

In [ ]:
# 可视化 Attention 计算图（简化版，展示数据流）

def trace_attention_graph():
    """模拟 ggml 构建 Attention 计算图的过程，对应 build_llama_graph 中的 attention 部分"""
    nodes = []
    edges = []
    
    def add_node(name, op=""):
        nodes.append({"name": name, "op": op})
        return len(nodes) - 1
    
    def add_edge(src, dst):
        edges.append((src, dst))
    
    # 构建 Attention 子图
    inp = add_node("cur (hidden)")
    attn_norm = add_node("attn_norm", "mul")
    rms = add_node("RMSNorm", "rms_norm")
    add_edge(inp, rms)
    add_edge(rms, attn_norm)
    
    wq = add_node("Wq (weight)")
    wk = add_node("Wk (weight)")
    wv = add_node("Wv (weight)")
    wo = add_node("Wo (weight)")
    
    Q = add_node("Q", "mul_mat")
    K = add_node("K", "mul_mat")
    V = add_node("V", "mul_mat")
    add_edge(attn_norm, Q); add_edge(wq, Q)
    add_edge(attn_norm, K); add_edge(wk, K)
    add_edge(attn_norm, V); add_edge(wv, V)
    
    Q_rope = add_node("Q_rope", "rope")
    K_rope = add_node("K_rope", "rope")
    add_edge(Q, Q_rope)
    add_edge(K, K_rope)
    
    kv_k = add_node("KV Cache K")
    kv_v = add_node("KV Cache V")
    add_edge(K_rope, kv_k)
    add_edge(V, kv_v)
    
    scores = add_node("QK^T/sqrt(d)", "mul_mat")
    add_edge(Q_rope, scores)
    add_edge(kv_k, scores)
    
    attn_w = add_node("attn_weights", "softmax")
    add_edge(scores, attn_w)
    
    attn_out = add_node("attn_output", "mul_mat")
    add_edge(attn_w, attn_out)
    add_edge(kv_v, attn_out)
    
    proj = add_node("output_proj", "mul_mat")
    add_edge(attn_out, proj)
    add_edge(wo, proj)
    
    residual = add_node("+ residual", "add")
    add_edge(proj, residual)
    add_edge(inp, residual)
    
    # 打印计算图
    for i, n in enumerate(nodes):
        in_edges = [e[0] for e in edges if e[1] == i]
        print(f"  [{i}] {n['name']:20s} {n['op']:10s}  <- {[nodes[e]['name'] for e in in_edges]}")
    
    print(f"\n总计 {len(nodes)} 个 ggml_tensor 节点，这只是一层 Attention")
    print(f"完整的 32 层 Transformer 大约 {len(nodes) * 32} 个节点 + FFN")

trace_attention_graph()


## 5. KV Cache：序列记忆的物理存储

### 5.1 llama.cpp 的 KV Cache 布局

```c
// llama.cpp 中的 KV Cache 数据结构
struct llama_kv_cache {
    bool has_shift;  // 是否支持 KV Cache 移位 (用于无限对话)
    
    uint32_t head;    // KV head 数量 (GQA 时可能与 Q head 不同)
    uint32_t size;    // 最大缓存 token 数 (等于 context_length)
    
    // 每个 Layer 有一组 K 和 V 缓存
    struct ggml_tensor * k_l[n_layer];  // shape: [n_embd_head_k, n_kv_head, size]
    struct ggml_tensor * v_l[n_layer];  // shape: [size, n_kv_head, n_embd_head_v]
    
    // 每个 cell (一个 token 位置的缓存) 的元数据
    struct llama_kv_cell {
        llama_pos pos;       // 该 cell 存储的序列位置 (-1 = 空闲)
        llama_seq_id seq_id; // 属于哪个序列
        bool has_seq_id;
    } cells[size];
};

// 关键操作：
// 1. kv_cache_find_slot()  — 找到一个空闲 cell 存放当前 token
// 2. ggml_cpy(K_new, k_l[layer].slice(pos))  — 将新 K 拷贝到正确位置
// 3. ggml_view_2d(k_l[layer], n_embd_head_k, n_past+1)  — 读取历史 K
```

### 5.2 llama.cpp KV Cache vs vLLM PagedAttention

| | llama.cpp | vLLM (PagedAttention) |
|------|-----------|----------------------|
| 分配策略 | 预分配连续 buffer，大小 = context_length × n_layer | 按需分配 block (16 tokens/block)，动态增长 |
| 碎片问题 | **内部碎片**：实际用 100 token 但分配了 4096 个位置 | **外部碎片**：block 可能分散，需要 block table 映射 |
| Cache 共享 | 无（每个序列独立） | **支持**：多个请求共享同一 system prompt 的 KV block |
| 实现复杂度 | 低（直接索引 `k_l[l][pos]`） | 高（block table 间接访问 `k_cache[block_table[i]]`） |

llama.cpp 的设计选择是务实的：**本地推理场景下并发少、序列少，预分配简简单单就够了**。

In [ ]:
# 模拟 llama.cpp KV Cache 的写入和读取
import numpy as np

class LlamaKVCache:
    """简化版 llama_kv_cache 模拟"""
    
    def __init__(self, n_layers: int, n_kv_heads: int, head_dim: int, max_tokens: int):
        self.n_layers = n_layers
        self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim
        self.max_tokens = max_tokens
        
        # 预分配连续 buffer: [n_kv_heads, max_tokens, head_dim]
        self.k_cache = np.zeros((n_layers, n_kv_heads, max_tokens, head_dim), dtype=np.float32)
        self.v_cache = np.zeros((n_layers, n_kv_heads, max_tokens, head_dim), dtype=np.float32)
        
        # Cell 元数据 (对应 llama_kv_cell)
        self.cell_pos = np.full(max_tokens, -1, dtype=np.int32)  # -1 = 空闲
        self.n_filled = 0
    
    def store(self, layer: int, pos: int, K_new: np.ndarray, V_new: np.ndarray):
        """对应 ggml_cpy(ctx, K, k_l[layer] 的 view)"""
        self.k_cache[layer, :, pos, :] = K_new  # [n_kv_heads, head_dim]
        self.v_cache[layer, :, pos, :] = V_new
        self.cell_pos[pos] = pos
        self.n_filled = max(self.n_filled, pos + 1)
    
    def get_history(self, layer: int) -> np.ndarray:
        """获取该层所有已缓存的 K values [n_kv_heads, n_filled, head_dim]"""
        return self.k_cache[layer, :, :self.n_filled, :]
    
    def usage(self) -> float:
        """缓存利用率"""
        return self.n_filled / self.max_tokens

# 演示
cache = LlamaKVCache(n_layers=32, n_kv_heads=8, head_dim=128, max_tokens=4096)

K_fake = np.random.randn(8, 128).astype(np.float32)
V_fake = np.random.randn(8, 128).astype(np.float32)

# 模拟生成 100 个 token
for pos in range(100):
    for layer in range(32):
        cache.store(layer, pos, K_fake, V_fake)

print(f"Stored {cache.n_filled} tokens in KV Cache")
print(f"Cache usage: {cache.usage()*100:.1f}%")
print(f"Memory per layer: {cache.k_cache[0].nbytes / 1024:.1f} KB")
print(f"Total KV Cache: {cache.k_cache.nbytes / 1024**2:.1f} MB")
print(f"\n这就是 llama.cpp KV Cache 的预分配模式——")
print(f"即使只用了 100/4096 个位置，显存也是按 max_tokens 分配的")


## 6. 后端抽象：一次写图，多端执行

llama.cpp 的后端抽象可能是它最有远见的设计。关键接口：

```c
// ggml/include/ggml-backend.h

// 后端接口 (类似 driver)
struct ggml_backend {
    ggml_backend_i iface;  // 虚函数表
    ggml_backend_context_t context;
    char name[64];
};

struct ggml_backend_i {
    const char *(*get_name)(ggml_backend_t backend);
    void (*free)(ggml_backend_t backend);
    
    // 内存管理
    ggml_backend_buffer_type_t (*get_default_buffer_type)(ggml_backend_t backend);
    
    // 同步
    bool (*supports_op)(ggml_backend_t backend, const struct ggml_tensor *op);
    
    // 关键：计算图执行
    ggml_backend_graph_copy_t (*graph_copy)(ggml_backend_t backend, 
                                            struct ggml_cgraph *graph);
    
    void (*synchronize)(ggml_backend_t backend);
};

// 四个主要后端实现：
// 1. CPU     — ggml/src/ggml.c               (scalar/AVX/NEON 优化)
// 2. CUDA    — ggml/src/ggml-cuda/           (cuBLAS + 自定义 kernel)
// 3. Metal   — ggml/src/ggml-metal/          (Apple GPU, MPS)
// 4. Vulkan  — ggml/src/ggml-vulkan/         (跨平台 GPU, 实验性)
```

### 后端调度的关键：ggml_backend_sched

```c
// 调度器决定计算图中每个节点在哪个后端执行
struct ggml_backend_sched {
    ggml_backend_t backends[GGML_MAX_BACKENDS];  // 优先级后端列表
    int n_backends;
    
    // ggml_backend_sched_graph_compute() 内部：
    //   遍历 cgraph 中所有节点
    //   对每个节点，按优先级查找支持该 op 的后端
    //   如果一个 backend 支持该 op 且 tensor 在上面 → 分配给它
    //   不支持 → 插入拷贝节点 → 分配回 CPU
};
```

**实际分割效果**（以 Metal 后端为例）：
```
整体计算图 (300+ 节点)
├── [Metal]  mul_mat 节点 (Q/K/V/O/FFN 投影)  ← GPU 计算
├── [Metal]  rope 节点                         ← GPU 计算
├── [Metal]  softmax 节点                       ← GPU 计算
├── [Metal]  silu / mul (FFN 激活)              ← GPU 计算
├── [CPU]    rms_norm 节点                      ← 小算子，留 CPU
├── [CPU]    cpy (KV Cache 存储)                ← 内存操作
└── [CPU]    reshape / view / permute            ← 不计算，零开销

插入的拷贝节点：
  CPU → GPU: embedding 输出、KV Cache 读取
  GPU → CPU: logits 输出（采样在 CPU 上做）
```

In [ ]:
# 模拟后端调度器的决策逻辑

class MockBackend:
    def __init__(self, name, supported_ops):
        self.name = name
        self.supported_ops = set(supported_ops)
    
    def supports(self, op):
        return op in self.supported_ops

gpu = MockBackend("Metal", {"mul_mat", "rope", "softmax", "silu", "mul"})
cpu = MockBackend("CPU", {"mul_mat", "rms_norm", "cpy", "view", "reshape", "add", "rope", "softmax", "silu", "mul"})

backends = [gpu, cpu]  # 按优先级排序

# 模拟 build_llama_graph 中创建的节点
ops_sequence = [
    ("embd", "mul_mat"),
    ("rms_norm_0", "rms_norm"),
    ("Q_proj", "mul_mat"),
    ("K_proj", "mul_mat"),
    ("V_proj", "mul_mat"),
    ("Q_rope", "rope"),
    ("K_rope", "rope"),
    ("kv_store", "cpy"),
    ("QK^T", "mul_mat"),
    ("softmax", "softmax"),
    ("attn_out", "mul_mat"),
    ("O_proj", "mul_mat"),
    ("residual", "add"),
    ("rms_norm_1", "rms_norm"),
    ("gate_proj", "mul_mat"),
    ("silu", "silu"),
    ("up_proj", "mul_mat"),
    ("gated", "mul"),
    ("down_proj", "mul_mat"),
]

print("Backend assignment (模拟 ggml_backend_sched):")
print(f"{'Node':15s} {'Op':10s} {'Backend':8s}")
print("-" * 35)

gpu_ops = 0
cpu_ops = 0
for name, op in ops_sequence:
    assigned = None
    for b in backends:
        if b.supports(op):
            assigned = b
            break
    if assigned:
        print(f"{name:15s} {op:10s} {assigned.name:8s}")
        if assigned.name == "Metal":
            gpu_ops += 1
        else:
            cpu_ops += 1

print(f"\nGPU (Metal): {gpu_ops} ops, CPU: {cpu_ops} ops")
print(f"GPU 占比: {gpu_ops/(gpu_ops+cpu_ops)*100:.0f}% (按节点数)")
print(f"\n计算量占比远高于节点数——")
print(f"12 个 mul_mat 节点占总 FLOPS 的 99%+，都在 GPU 上执行")


## 7. 关键设计决策与权衡

总结 llama.cpp 架构中最值得理解的几个选择：

### 为什么自研 ggml 而不是用 ONNX Runtime / TensorFlow Lite？

1. **量化是核心，不能是后加的**：Q4_K_M 不是「把 FP32 矩阵量化成 INT4」这么简单。
   ggml 的量化是与计算图**深度融合**的——矩阵乘法算子知道输入是量化的，直接在量化域计算。
   外部 runtime 做不到这点。

2. **mmap 加载是刚需**：GGUF + mmap = 「点开即用」。对本地用户来说，
   等 10 秒加载 4GB 模型 vs 等 2 分钟把模型拷到 GPU 显存再解析——体验天差地别。

3. **控制粒度**：自研意味着每个算子的实现都可以针对量化权重优化。
   例如 `ggml_compute_forward_mul_mat_q4_K_f32` 这样高度特化的函数。

### 为什么 KV Cache 不学 vLLM 的 PagedAttention？

因为场景不同：
- llama.cpp 面向**个人电脑**，通常 1-2 个并发请求，不需要复杂的块分配
- 预分配简单高效，CPU 上的指针运算 `k_l[l][pos]` 比 block_table 间接寻址快
- 社区有人在做 llama.cpp 的 PagedAttention 实现（实验性），但不作为默认

### CPU 推理为什么能这么快？

ggml 的 CPU 后端有三板斧：
1. **多线程并行**：token generation 阶段在 n_embd 维度并行（每个线程算一部分注意力头）
2. **SIMD 手写内核**：x86 用 AVX2/AVX-512，ARM 用 NEON——都是手写 intrinsics
3. **量化后的矩阵乘法极快**：4-bit 权重 × FP32 激活向量 → 内存带宽已经成了瓶颈，不是计算

## 下一步

- → `02-ollama-architecture.ipynb`：Ollama 在 llama.cpp 之上加了什么？Modelfile、调度层、API 服务
- → 返回 `../00-overview.ipynb` 查看服务端框架（vLLM, TensorRT-LLM）

## 关键源码文件索引

| 想看什么 | 看哪个文件 | 看哪个函数 |
|---------|----------|----------|
| GGUF 文件格式 | `gguf/gguf.py` | `GGUFReader.__init__` |
| 计算图构建 | `llama.cpp` | `build_llama_graph()` |
| K-quant 反量化 | `ggml/src/ggml-quants.c` | `ggml_dequantize_q4_K()` |
| CPU 矩阵乘法 | `ggml/src/ggml.c` | `ggml_compute_forward_mul_mat()` |
| CUDA 后端 | `ggml/src/ggml-cuda/` | `ggml_cuda_mul_mat()` |
| Metal 后端 | `ggml/src/ggml-metal/` | Metal Shading Language kernels |
| 推理主循环 | `llama.cpp` | `llama_decode_internal()` |
| KV Cache 管理 | `llama.cpp` | `llama_kv_cache_find_slot()` |
| 后端调度 | `ggml/src/ggml-backend.c` | `ggml_backend_sched_graph_compute()` |